In [1]:
import pandas as pd
import numpy as np
import joblib
import time

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

In [2]:
X_train = joblib.load("../data/X_train_zero_day.pkl")
y_train = joblib.load("../data/y_train_zero_day.pkl")

X_test_seen = joblib.load("../data/X_test_seen.pkl")
y_test_seen = joblib.load("../data/y_test_seen.pkl")

X_test_zero_day = joblib.load("../data/X_test_zero_day.pkl")
y_test_zero_day = joblib.load("../data/y_test_zero_day.pkl")

In [3]:
X_train_scaled = joblib.load(
    "../data/X_train_zero_day_scaled.pkl"
)

X_test_seen_scaled = joblib.load(
    "../data/X_test_seen_scaled.pkl"
)

X_test_zero_day_scaled = joblib.load(
    "../data/X_test_zero_day_scaled.pkl"
)

In [4]:
print("===== DATASET SHAPES =====")

print("X_train          :", X_train.shape)
print("y_train          :", y_train.shape)

print("X_test_seen      :", X_test_seen.shape)
print("y_test_seen      :", y_test_seen.shape)

print("X_test_zero_day  :", X_test_zero_day.shape)
print("y_test_zero_day  :", y_test_zero_day.shape)

print("\n===== LABEL DISTRIBUTION =====")

print("Train:")
print(y_train.value_counts())

print("\nTest Seen:")
print(y_test_seen.value_counts())

print("\nTest Zero-Day:")
print(y_test_zero_day.value_counts())

===== DATASET SHAPES =====
X_train          : (125973, 122)
y_train          : (125973,)
X_test_seen      : (18794, 122)
y_test_seen      : (18794,)
X_test_zero_day  : (3750, 122)
y_test_zero_day  : (3750,)

===== LABEL DISTRIBUTION =====
Train:
binary_label
0    67343
1    58630
Name: count, dtype: int64

Test Seen:
binary_label
0    9711
1    9083
Name: count, dtype: int64

Test Zero-Day:
binary_label
1    3750
Name: count, dtype: int64


In [5]:
def evaluate_zero_day_model(
    model,
    X_train,
    y_train,
    X_test_seen,
    y_test_seen,
    X_test_zero_day,
    y_test_zero_day
):
    
    print("=" * 60)
    print(f"Model: {model.__class__.__name__}")
    print("=" * 60)

    # -------------------------
    # Training
    # -------------------------
    start_train = time.time()

    model.fit(X_train, y_train)

    train_time = time.time() - start_train

    # -------------------------
    # Prediction - Seen Test
    # -------------------------
    start_predict_seen = time.time()

    y_pred_seen = model.predict(X_test_seen)

    predict_seen_time = time.time() - start_predict_seen

    # -------------------------
    # Prediction - Zero-Day
    # -------------------------
    start_predict_zero = time.time()

    y_pred_zero_day = model.predict(X_test_zero_day)

    predict_zero_day_time = time.time() - start_predict_zero

    # -------------------------
    # Normal Test Metrics
    # -------------------------
    accuracy = accuracy_score(
        y_test_seen,
        y_pred_seen
    )

    precision = precision_score(
        y_test_seen,
        y_pred_seen
    )

    recall = recall_score(
        y_test_seen,
        y_pred_seen
    )

    f1 = f1_score(
        y_test_seen,
        y_pred_seen
    )

    # -------------------------
    # Zero-Day Detection Rate
    # -------------------------
    zero_day_detection_rate = np.mean(
        y_pred_zero_day == 1
    )

    print("\n--- Seen Test Performance ---")

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-Score : {f1:.4f}")

    print("\n--- Zero-Day Performance ---")

    print(
        f"Zero-Day Detection Rate: "
        f"{zero_day_detection_rate:.4f}"
    )

    print(
        f"Zero-Day Detection Rate: "
        f"{zero_day_detection_rate * 100:.2f}%"
    )

    print("\n--- Time ---")

    print(
        f"Training Time: "
        f"{train_time:.4f} seconds"
    )

    print(
        f"Seen Prediction Time: "
        f"{predict_seen_time:.4f} seconds"
    )

    print(
        f"Zero-Day Prediction Time: "
        f"{predict_zero_day_time:.4f} seconds"
    )

    return {
        "model": model.__class__.__name__,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "zero_day_detection_rate":
            zero_day_detection_rate,
        "train_time": train_time,
        "predict_seen_time":
            predict_seen_time,
        "predict_zero_day_time":
            predict_zero_day_time
    }

Decsision Tree

In [6]:
dt = DecisionTreeClassifier(
    random_state=42
)

In [7]:
dt_result = evaluate_zero_day_model(
    dt,
    X_train,
    y_train,
    X_test_seen,
    y_test_seen,
    X_test_zero_day,
    y_test_zero_day
)

Model: DecisionTreeClassifier



--- Seen Test Performance ---
Accuracy : 0.8656
Precision: 0.9576
Recall   : 0.7555
F1-Score : 0.8446

--- Zero-Day Performance ---
Zero-Day Detection Rate: 0.3827
Zero-Day Detection Rate: 38.27%

--- Time ---
Training Time: 4.3958 seconds
Seen Prediction Time: 0.0157 seconds
Zero-Day Prediction Time: 0.0033 seconds


Random Forest

In [8]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [9]:
rf_result = evaluate_zero_day_model(
    rf,
    X_train,
    y_train,
    X_test_seen,
    y_test_seen,
    X_test_zero_day,
    y_test_zero_day
)

Model: RandomForestClassifier

--- Seen Test Performance ---
Accuracy : 0.8737
Precision: 0.9633
Recall   : 0.7680
F1-Score : 0.8546

--- Zero-Day Performance ---
Zero-Day Detection Rate: 0.2203
Zero-Day Detection Rate: 22.03%

--- Time ---
Training Time: 10.4435 seconds
Seen Prediction Time: 0.1053 seconds
Zero-Day Prediction Time: 0.0418 seconds


KNN

In [10]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    n_jobs=-1
)

knn_result = evaluate_zero_day_model(
    knn,
    X_train_scaled,
    y_train,
    X_test_seen_scaled,
    y_test_seen,
    X_test_zero_day_scaled,
    y_test_zero_day
)

Model: KNeighborsClassifier

--- Seen Test Performance ---
Accuracy : 0.8373
Precision: 0.9065
Recall   : 0.7397
F1-Score : 0.8147

--- Zero-Day Performance ---
Zero-Day Detection Rate: 0.4352
Zero-Day Detection Rate: 43.52%

--- Time ---
Training Time: 0.0383 seconds
Seen Prediction Time: 18.0744 seconds
Zero-Day Prediction Time: 3.4507 seconds


Linear SVM

In [11]:
svm = LinearSVC(
    random_state=42,
    max_iter=5000
)

svm_result = evaluate_zero_day_model(
    svm,
    X_train_scaled,
    y_train,
    X_test_seen_scaled,
    y_test_seen,
    X_test_zero_day_scaled,
    y_test_zero_day
)

Model: LinearSVC

--- Seen Test Performance ---
Accuracy : 0.8218
Precision: 0.8990
Recall   : 0.7111
F1-Score : 0.7941

--- Zero-Day Performance ---
Zero-Day Detection Rate: 0.3653
Zero-Day Detection Rate: 36.53%

--- Time ---
Training Time: 19.3747 seconds
Seen Prediction Time: 0.0183 seconds
Zero-Day Prediction Time: 0.0013 seconds


Logistic Regression

In [12]:
lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

lr_result = evaluate_zero_day_model(
    lr,
    X_train_scaled,
    y_train,
    X_test_seen_scaled,
    y_test_seen,
    X_test_zero_day_scaled,
    y_test_zero_day
)

Model: LogisticRegression


c:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



--- Seen Test Performance ---
Accuracy : 0.8226
Precision: 0.8997
Recall   : 0.7123
F1-Score : 0.7951

--- Zero-Day Performance ---
Zero-Day Detection Rate: 0.4064
Zero-Day Detection Rate: 40.64%

--- Time ---
Training Time: 5.5333 seconds
Seen Prediction Time: 0.0073 seconds
Zero-Day Prediction Time: 0.0013 seconds
